In [ ]:
from PIL import Image

import torch
import torch.nn as nn 
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchvision
import torchvision.transforms as transforms
from torchvision.transforms.v2 import MixUp, CutMix, RandomChoice
from CNN import ImageNeuralNetwork

In [ ]:
best_accuracy = 61.28

def check_accuracy(cnn):
    global best_accuracy 
    correct = 0
    total = 0
    cnn.eval()

    with torch.no_grad(): 
        for data in test_loader:
            images, labels = data
            
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = cnn(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    cnn.train()
    accuracy = 100 * correct / total
    print(f'Accuracy: {accuracy}%')
    if(accuracy > best_accuracy):
        best_accuracy = accuracy
        torch.save(cnn.state_dict(), f'trained_net_{accuracy}.pth')
    
def load_image(image_path, new_transform):
    image = Image.open(image_path).convert('RGB')
    image = new_transform(image)
    image = image.unsqueeze(0)
    return image

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(64, padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.RandAugment(num_ops=2, magnitude=9),  
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4802, 0.4481, 0.3975], std=[0.2770, 0.2691, 0.2821]),  
    transforms.RandomErasing(p=0.25),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4802, 0.4481, 0.3975], std=[0.2770, 0.2691, 0.2821]),
])

In [ ]:
train_data = torchvision.datasets.ImageFolder(root='data/tiny-imagenet-200/train', transform=train_transform)
test_data = torchvision.datasets.ImageFolder(root='data/tiny-imagenet-200/val', transform=test_transform
)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=512, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=512, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True)

In [ ]:
num_epochs = 350
net = ImageNeuralNetwork(64, 4, 6, 200).to(device)
loss_function = nn.CrossEntropyLoss(label_smoothing = 0.1)
optimizer = optim.SGD(net.parameters(), lr = 0.1, momentum = 0.9, weight_decay = 1e-4, nesterov = True)
scheduler = CosineAnnealingLR(optimizer, T_max = num_epochs, eta_min = 1e-7)

In [ ]:
mixup = MixUp(alpha=0.2, num_classes=200)
cutmix = CutMix(alpha=1.0, num_classes=200)
mixup_cutmix = RandomChoice([mixup, cutmix])

In [ ]:
NUM_NO_MIX = 35
for epoch in range(1, num_epochs + 1):
    print(f'Training epoch {epoch}...')
    
    running_loss = 0.0
    
    for i, data in enumerate(train_loader):
        inputs, labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)

        if epoch <= num_epochs - NUM_NO_MIX:
            inputs, labels = mixup_cutmix(inputs, labels)

        optimizer.zero_grad()
        outputs = net(inputs)

        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    print(f'Loss: {running_loss / len(train_loader):.4f}, LR: {current_lr:.6f}')
    
    if(epoch >= 200 and epoch % 25 == 0):
        check_accuracy(net)

In [ ]:
net = ImageNeuralNetwork(64, 4, 6, 200).to(device)
net.load_state_dict(torch.load(f'trained_net_{best_accuracy}.pth'))

In [ ]:
new_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4802, 0.4481, 0.3975], std=[0.2770, 0.2691, 0.2821]),
])

In [ ]:
image_paths = ['uni.jpg']
images = [load_image(img, new_transform) for img in image_paths]
net.eval()
with torch.no_grad():
    for image in images:
        outputs = net(image.to(device))
        _, predicted = torch.max(outputs, 1)
        print(f'Prediction: {train_data.classes[predicted.item()]}')